In [ ]:
import pandas as pd
import sqlite3
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

In [ ]:
db_path = r"../data/database/faers_2025.db"
conn = sqlite3.connect(db_path)
query_demo = "SELECT primaryid, age, sex, wt, rept_cod, occp_cod FROM demo_clean"
df_master_a = pd.read_sql_query(query_demo, conn)
df_master_a.drop_duplicates(subset=['primaryid'], inplace=True)

In [ ]:
df_master_a['rept_cod'] = df_master_a['rept_cod'].astype(str).str.upper().str.strip().replace('NAN', 'UNK')
df_master_a['occp_cod'] = df_master_a['occp_cod'].astype(str).str.upper().str.strip().replace('NAN', 'UNK')

In [ ]:
df_outc = pd.read_sql_query("SELECT primaryid, outc_cod FROM outc_clean", conn)
df_outc['is_severe'] = df_outc['outc_cod'].isin(['DE', 'HO', 'LT']).astype(int)
target_df = df_outc.groupby('primaryid')['is_severe'].max().reset_index()

df_master_a = df_master_a.merge(target_df, on='primaryid', how='inner')

In [ ]:
df_drug = pd.read_sql_query("SELECT primaryid, role_cod, final_drug_name, route, dechal FROM drug_clean", conn)

df_drug['route'] = df_drug['route'].astype(str).str.upper().str.strip().replace('NAN', 'UNK')

polypharmacy = df_drug.groupby('primaryid').size().reset_index(name='num_drugs')

df_drug['positive_dechal'] = (df_drug['dechal'] == 'Y').astype(int)
dechal_feat = df_drug.groupby('primaryid')['positive_dechal'].max().reset_index(name='has_positive_dechallenge')

primary_drugs = df_drug[df_drug['role_cod'] == 'PS'].copy()

ps_features = primary_drugs.groupby('primaryid').agg({
    'final_drug_name': 'first',
    'route': 'first'
}).reset_index()
ps_features.rename(columns={'final_drug_name': 'primary_suspect_drug', 'route': 'ps_route'}, inplace=True)

In [ ]:
df_ther = pd.read_sql_query("SELECT primaryid, dur FROM ther_clean WHERE dur IS NOT NULL", conn)
duration_feat = df_ther.groupby('primaryid')['dur'].max().reset_index(name='max_therapy_duration')

df_indi = pd.read_sql_query("SELECT primaryid FROM indi_clean", conn)
comorbidities = df_indi.groupby('primaryid').size().reset_index(name='num_comorbidities')

In [ ]:
features = [polypharmacy, dechal_feat, ps_features, duration_feat, comorbidities]

for feat_df in features:
    df_master_a = df_master_a.merge(feat_df, on='primaryid', how='left')


df_master_a['primary_suspect_drug'] = df_master_a['primary_suspect_drug'].fillna('UNKNOWN_DRUG')
df_master_a['ps_route'] = df_master_a['ps_route'].fillna('UNK')


fillna_cols = ['num_drugs', 'has_positive_dechallenge', 'num_comorbidities']
df_master_a[fillna_cols] = df_master_a[fillna_cols].fillna(0)

In [ ]:
print(f"\n Matrix A (V2) is complete and fully standardized!")
print(f"Total Usable Patient Records: {len(df_master_a):,}")
print(f"Matrix V2 Columns: {list(df_master_a.columns)}")

df_master_a.to_sql('matrix_a_outcome_v2', conn, if_exists='replace', index=False)
conn.close()

In [ ]:
db_path = r"../data/database/faers_2025.db"
conn = sqlite3.connect(db_path)
df = pd.read_sql_query("SELECT * FROM matrix_a_outcome_v2", conn)
conn.close()

In [ ]:
df.drop(columns=['primaryid'], inplace=True)

X = df.drop(columns=['is_severe'])
y = df['is_severe']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [ ]:
target_cols = ['primary_suspect_drug', 'ps_route', 'rept_cod', 'occp_cod']
overall_mean = y_train.mean()

for col in target_cols:

    target_means = y_train.groupby(X_train[col]).mean()

    X_train[col + '_encoded'] = X_train[col].map(target_means)
    X_test[col + '_encoded'] = X_test[col].map(target_means)
    

    X_train[col + '_encoded'] = X_train[col + '_encoded'].fillna(overall_mean)
    X_test[col + '_encoded'] = X_test[col + '_encoded'].fillna(overall_mean)
    

    X_train.drop(columns=[col], inplace=True)
    X_test.drop(columns=[col], inplace=True)

X_train = pd.get_dummies(X_train, columns=['sex'], drop_first=False)
X_test = pd.get_dummies(X_test, columns=['sex'], drop_first=False)

X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [ ]:
neg_class_count = (y_train == 0).sum()
pos_class_count = (y_train == 1).sum()
scale_weight = neg_class_count / pos_class_count

xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    scale_pos_weight=scale_weight,
    max_depth=7,                
    learning_rate=0.05,          
    n_estimators=300,             
    random_state=42,
    tree_method='hist',
    enable_categorical=False 
)

xgb_model.fit(X_train, y_train)

In [ ]:
y_pred = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]


print(classification_report(y_test, y_pred))

auc_score = roc_auc_score(y_test, y_pred_proba)
print(f" ROC-AUC Score: {auc_score:.4f}\n")

In [ ]:
feature_importances = pd.Series(xgb_model.feature_importances_, index=X_train.columns)
top_features = feature_importances.sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 7))
sns.barplot(x=top_features.values, y=top_features.index, palette='magma')
plt.title('V2 XGBoost Feature Importance (Target Encoded)', fontsize=16, pad=15)
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import sqlite3
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import optuna


# 1. Load the Data (Assuming Matrix A V2 is ready)
db_path = r"../data/database/faers_2025.db"
conn = sqlite3.connect(db_path)
df = pd.read_sql_query("SELECT * FROM matrix_a_outcome_v2", conn)
conn.close()

# Drop ID
df.drop(columns=['primaryid'], inplace=True)

# Define X and y
X = df.drop(columns=['is_severe'])
y = df['is_severe']

# Train/Test Split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 2. Target Encoding (Using the fixed future-proof method)
target_cols = ['primary_suspect_drug', 'ps_route', 'rept_cod', 'occp_cod']
overall_mean = y_train.mean()

for col in target_cols:
    target_means = y_train.groupby(X_train[col]).mean()
    X_train[col + '_encoded'] = X_train[col].map(target_means)
    X_test[col + '_encoded'] = X_test[col].map(target_means)
    
    X_train[col + '_encoded'] = X_train[col + '_encoded'].fillna(overall_mean)
    X_test[col + '_encoded'] = X_test[col + '_encoded'].fillna(overall_mean)
    
    X_train.drop(columns=[col], inplace=True)
    X_test.drop(columns=[col], inplace=True)

X_train = pd.get_dummies(X_train, columns=['sex'], drop_first=False)
X_test = pd.get_dummies(X_test, columns=['sex'], drop_first=False)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

# Calculate class weight
neg_class_count = (y_train == 0).sum()
pos_class_count = (y_train == 1).sum()
scale_weight = neg_class_count / pos_class_count

# 3. Define the Optuna Objective Function
def objective(trial):
    # Suggest values for the hyperparameters
    param = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'hist',
        'scale_pos_weight': scale_weight,
        'random_state': 42,
        'enable_categorical': False,
        
        # Hyperparameters to tune
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10)
    }

    # Train model
    model = xgb.XGBClassifier(**param)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

    # Predict and calculate ROC-AUC
    preds_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, preds_proba)
    
    return auc


# Note: n_trials=20 is a good starting point. You can increase it for better results, but it takes longer.
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

print(f"Best Trial: {study.best_trial.number}")
print(f"Best ROC-AUC Value: {study.best_value:.4f}")
print("Best Parameters:")
for key, value in study.best_trial.params.items():
    print(f"    {key}: {value}")

In [ ]:
db_path = r"../data/database/faers_2025.db"
conn = sqlite3.connect(db_path)

df_v2 = pd.read_sql_query("SELECT * FROM matrix_a_outcome_v2", conn)

print("2. Extracting & Scoring Symptoms (Target Encoding)...")

df_reac = pd.read_sql_query("SELECT primaryid, pt FROM reac_clean", conn)

df_outc = pd.read_sql_query("SELECT primaryid, outc_cod FROM outc_clean", conn)
df_outc['is_severe'] = df_outc['outc_cod'].isin(['DE', 'HO', 'LT']).astype(int)
target_df = df_outc.groupby('primaryid')['is_severe'].max().reset_index()


df_symptoms = df_reac.merge(target_df, on='primaryid', how='inner')


symptom_scores = df_symptoms.groupby('pt')['is_severe'].mean().reset_index(name='symptom_risk_score')


df_symptoms = df_symptoms.merge(symptom_scores, on='pt', how='left')


patient_symptoms = df_symptoms.groupby('primaryid').agg(
    avg_symptom_risk=('symptom_risk_score', 'mean'),
    max_symptom_risk=('symptom_risk_score', 'max'),
    num_symptoms=('pt', 'count')
).reset_index()



df_master_a_v3 = df_v2.merge(patient_symptoms, on='primaryid', how='left')

global_mean_risk = symptom_scores['symptom_risk_score'].mean()
df_master_a_v3['avg_symptom_risk'] = df_master_a_v3['avg_symptom_risk'].fillna(global_mean_risk)
df_master_a_v3['max_symptom_risk'] = df_master_a_v3['max_symptom_risk'].fillna(global_mean_risk)
df_master_a_v3['num_symptoms'] = df_master_a_v3['num_symptoms'].fillna(0)

print(f"\n Matrix A (V3) is complete!")
print(f"Total Usable Patient Records: {len(df_master_a_v3):,}")
print(f"Matrix V3 Columns: {list(df_master_a_v3.columns)}")

df_master_a_v3.to_sql('matrix_a_outcome_v3', conn, if_exists='replace', index=False)
conn.close()

In [ ]:
db_path = r"../data/database/faers_2025.db"
conn = sqlite3.connect(db_path)
df = pd.read_sql_query("SELECT * FROM matrix_a_outcome_v3", conn)
conn.close()


df.drop(columns=['primaryid'], inplace=True)

X = df.drop(columns=['is_severe'])
y = df['is_severe']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training Set: {X_train.shape[0]:,} records")


print("⚙️ Applying Target Encoding to clinical text features...")
target_cols = ['primary_suspect_drug', 'ps_route', 'rept_cod', 'occp_cod']
overall_mean = y_train.mean()

for col in target_cols:
    target_means = y_train.groupby(X_train[col]).mean()
    X_train[col + '_encoded'] = X_train[col].map(target_means)
    X_test[col + '_encoded'] = X_test[col].map(target_means)
    
    X_train[col + '_encoded'] = X_train[col + '_encoded'].fillna(overall_mean)
    X_test[col + '_encoded'] = X_test[col + '_encoded'].fillna(overall_mean)
    
    X_train.drop(columns=[col], inplace=True)
    X_test.drop(columns=[col], inplace=True)


X_train = pd.get_dummies(X_train, columns=['sex'], drop_first=False)
X_test = pd.get_dummies(X_test, columns=['sex'], drop_first=False)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

neg_class_count = (y_train == 0).sum()
pos_class_count = (y_train == 1).sum()
scale_weight = neg_class_count / pos_class_count


print("\n🧠 Training Ultimate XGBoost Model with Optuna Parameters...")
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    scale_pos_weight=scale_weight,
    max_depth=9,                          # From Optuna
    learning_rate=0.12314830979536133,    # From Optuna
    n_estimators=450,                     # From Optuna
    subsample=0.7975405105297657,         # From Optuna
    colsample_bytree=0.6961095979468281,  # From Optuna
    min_child_weight=5,                   # From Optuna
    random_state=42,
    tree_method='hist',
    enable_categorical=False 
)

xgb_model.fit(X_train, y_train)


y_pred = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]


print(classification_report(y_test, y_pred))

auc_score = roc_auc_score(y_test, y_pred_proba)
print(f" ULTIMATE ROC-AUC Score: {auc_score:.4f}\n")


feature_importances = pd.Series(xgb_model.feature_importances_, index=X_train.columns)
top_features = feature_importances.sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 7))
sns.barplot(x=top_features.values, y=top_features.index, palette='crest')
plt.title('V3 Ultimate XGBoost Feature Importance', fontsize=16, pad=15)
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.show()